# Proxy5 — Deep Learning Model Comparison for Oil Recovery Prediction

**Objective:** Compare four deep learning architectures for predicting cumulative oil recovery (RF%) from reservoir simulation proxy data.

| # | Model | Key Strength |
|---|-------|--------------|
| 1 | **MLP** — Multi-Layer Perceptron | Universal function approximator; fast baseline |
| 2 | **LSTM** — Long Short-Term Memory | Captures temporal/sequential dependencies |
| 3 | **CNN-LSTM** — 1-D Conv + LSTM | Local feature extraction + temporal memory |
| 4 | **PINN** — Physics-Informed Neural Network | Embeds Buckley–Leverett PDE as a physics loss |

**Physics reference:** Liu et al. *Physics of Fluids* 37 036622 (2025) — 1-D two-phase BL equation  
**Data:** Synthetic reservoir proxy generated from Buckley–Leverett + Corey rel-perms (CMG-consistent parameters)

## 1. Imports & Reproducibility

In [ ]:
import warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.integrate import solve_ivp

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('tab10')

COLORS = {'MLP': '#1f77b4', 'LSTM': '#ff7f0e', 'CNN-LSTM': '#2ca02c', 'PINN': '#d62728'}

## 2. Synthetic Data Generation

Proxy data is generated using a 1-D Buckley–Leverett model with Corey relative permeabilities.
Each sample represents a unique reservoir/injection scenario (varying porosity, viscosity ratio, injection rate, polymer concentration, rel-perm exponents). The target is cumulative oil recovery factor at multiple time steps.

In [ ]:
# --------------------------------------------------------------------------
# Physical constants — Pelican Lake HP-6 Pilot baseline (pinn_polymer_flood.py)
# --------------------------------------------------------------------------
SWC    = 0.30   # connate water saturation
SOR    = 0.20   # residual oil saturation
KRW0   = 0.10   # endpoint krw
KRO0   = 1.00   # endpoint kro
MU_O   = 1650.0 # oil viscosity [cp]

def corey_krw(Sw, Swc=SWC, Sor=SOR, krw0=KRW0, nw=3.0):
    Se = np.clip((Sw - Swc) / (1 - Swc - Sor), 0, 1)
    return krw0 * Se**nw

def corey_kro(Sw, Swc=SWC, Sor=SOR, kro0=KRO0, no=2.2):
    Se = np.clip((1 - Sw - Sor) / (1 - Swc - Sor), 0, 1)
    return kro0 * Se**no

def water_fractional_flow(Sw, mu_w, mu_o=MU_O, nw=3.0, no=2.2):
    krw = corey_krw(Sw, nw=nw)
    kro = corey_kro(Sw, no=no)
    mob_w = krw / mu_w
    mob_o = kro / mu_o
    denom = mob_w + mob_o
    return np.where(denom > 1e-12, mob_w / denom, 0.0)

def compute_recovery_curve(phi, nw, no, mu_w, vp_max=2.0, n_pv=50):
    """Returns (pv_injected, recovery_factor) arrays via fractional-flow theory."""
    # Sw grid for fw curve
    Sw_grid = np.linspace(SWC + 1e-4, 1 - SOR - 1e-4, 300)
    fw_grid = water_fractional_flow(Sw_grid, mu_w, nw=nw, no=no)
    dfw_dSw = np.gradient(fw_grid, Sw_grid)

    # Breakthrough Sw: tangent from (Swc, 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        tangent = np.where(Sw_grid > SWC + 1e-3,
                           (fw_grid - 0.0) / (Sw_grid - SWC), 0.0)
    Sw_bt_idx = np.argmax(tangent)
    Sw_bt = Sw_grid[Sw_bt_idx]
    fw_bt = fw_grid[Sw_bt_idx]

    # Build recovery vs PVI using Welge construction
    pv_inj = np.linspace(0, vp_max, n_pv)
    RF = np.zeros(n_pv)
    OOIP = (1 - SWC - SOR)  # dimensionless OOIP per unit PV

    for i, pvi in enumerate(pv_inj):
        if pvi == 0:
            RF[i] = 0.0
            continue
        # pre-breakthrough: no water prod
        pvi_bt = 1.0 / (dfw_dSw[Sw_bt_idx] + 1e-12)
        if pvi < pvi_bt:
            RF[i] = pvi * (1 - SWC) / (1 - SWC - SOR) * fw_bt
        else:
            # post-breakthrough: Sw at producer moves along fw curve
            slope_needed = 1.0 / pvi
            post_idx = Sw_bt_idx + np.argmin(np.abs(dfw_dSw[Sw_bt_idx:] - slope_needed))
            Sw_prod = Sw_grid[min(post_idx, len(Sw_grid)-1)]
            fw_prod = fw_grid[min(post_idx, len(fw_grid)-1)]
            Sw_avg = Sw_prod + (1 - fw_prod) * pvi
            Np = Sw_avg - SWC  # dimensionless cumulative oil produced
            RF[i] = np.clip(Np / OOIP, 0, 1)

    return pv_inj, np.clip(RF, 0, 1)

# --------------------------------------------------------------------------
# Generate dataset: N_CASES scenarios × N_TIMESTEP time steps
# --------------------------------------------------------------------------
N_CASES = 600
N_TIMESTEP = 50
VPM = 2.0

rng = np.random.default_rng(SEED)

phi_arr   = rng.uniform(0.20, 0.40, N_CASES)       # porosity
nw_arr    = rng.uniform(1.5,  4.5,  N_CASES)       # Corey water exponent
no_arr    = rng.uniform(1.5,  3.5,  N_CASES)       # Corey oil exponent
muw_arr   = rng.uniform(0.5,  10.0, N_CASES)       # water (polymer) viscosity [cp]
perm_arr  = rng.uniform(50,   2000, N_CASES)       # permeability [md]
thick_arr = rng.uniform(5,    30,   N_CASES)       # net pay [m]

pv_inj_ref = np.linspace(0, VPM, N_TIMESTEP)

records = []
for i in range(N_CASES):
    _, rf_curve = compute_recovery_curve(phi_arr[i], nw_arr[i], no_arr[i],
                                          muw_arr[i], vp_max=VPM, n_pv=N_TIMESTEP)
    # Add small noise to simulate measurement uncertainty
    rf_curve = np.clip(rf_curve + rng.normal(0, 0.005, N_TIMESTEP), 0, 1)
    for t in range(N_TIMESTEP):
        records.append({
            'case':      i,
            'pv_inj':    pv_inj_ref[t],
            'phi':       phi_arr[i],
            'nw':        nw_arr[i],
            'no':        no_arr[i],
            'mu_w':      muw_arr[i],
            'permeability': perm_arr[i],
            'net_pay':   thick_arr[i],
            'viscosity_ratio': MU_O / muw_arr[i],
            'RF':        rf_curve[t]
        })

df = pd.DataFrame(records)
print(f'Dataset shape: {df.shape}')
print(df.describe().round(4))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ----- 3.1  RF distribution & sample recovery curves ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# RF distribution
axes[0].hist(df['RF'], bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Recovery Factor (RF)')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution — RF')

# Sample recovery curves
sample_cases = rng.choice(N_CASES, 30, replace=False)
for c in sample_cases:
    sub = df[df['case'] == c]
    axes[1].plot(sub['pv_inj'], sub['RF'], alpha=0.4, linewidth=0.8, color='steelblue')
axes[1].set_xlabel('Pore Volumes Injected')
axes[1].set_ylabel('Recovery Factor')
axes[1].set_title('Sample RF Curves (30 cases)')

# RF vs viscosity ratio
end_rf = df[df['pv_inj'] == df['pv_inj'].max()]
sc = axes[2].scatter(end_rf['viscosity_ratio'], end_rf['RF'],
                     c=end_rf['nw'], cmap='viridis', alpha=0.6, s=15)
plt.colorbar(sc, ax=axes[2], label='nw (Corey water exp.)')
axes[2].set_xlabel('Viscosity Ratio (μo/μw)')
axes[2].set_ylabel('Final RF')
axes[2].set_title('Final RF vs Viscosity Ratio')
axes[2].set_xscale('log')

plt.suptitle('Exploratory Data Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('plot_01_eda.png', bbox_inches='tight')
plt.show()
print('Saved plot_01_eda.png')

In [ ]:
# ----- 3.2  Correlation heatmap ---
feat_cols = ['pv_inj', 'phi', 'nw', 'no', 'mu_w', 'permeability',
             'net_pay', 'viscosity_ratio', 'RF']
corr = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('plot_02_correlation.png', bbox_inches='tight')
plt.show()
print('Saved plot_02_correlation.png')

In [ ]:
# ----- 3.3  Feature distributions ---
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
features = ['pv_inj', 'phi', 'nw', 'no', 'mu_w', 'permeability', 'net_pay', 'viscosity_ratio']
for ax, feat in zip(axes.flat, features):
    ax.hist(df[feat], bins=40, color='teal', edgecolor='white', linewidth=0.3, alpha=0.8)
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
plt.suptitle('Input Feature Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('plot_03_features.png', bbox_inches='tight')
plt.show()
print('Saved plot_03_features.png')

## 4. Data Preprocessing

In [ ]:
FEATURE_COLS = ['pv_inj', 'phi', 'nw', 'no', 'mu_w',
                'permeability', 'net_pay', 'viscosity_ratio']
TARGET_COL   = 'RF'

# Train/val/test split by CASE (no temporal leakage)
cases = np.arange(N_CASES)
train_cases, temp_cases = train_test_split(cases, test_size=0.30, random_state=SEED)
val_cases,   test_cases = train_test_split(temp_cases, test_size=0.50, random_state=SEED)

df_train = df[df['case'].isin(train_cases)].reset_index(drop=True)
df_val   = df[df['case'].isin(val_cases)].reset_index(drop=True)
df_test  = df[df['case'].isin(test_cases)].reset_index(drop=True)

print(f'Train cases: {len(train_cases)} | Val: {len(val_cases)} | Test: {len(test_cases)}')
print(f'Train rows:  {len(df_train)}    | Val: {len(df_val)}   | Test: {len(df_test)}')

# Scale features
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train = scaler_X.fit_transform(df_train[FEATURE_COLS])
y_train = scaler_y.fit_transform(df_train[[TARGET_COL]])
X_val   = scaler_X.transform(df_val[FEATURE_COLS])
y_val   = scaler_y.transform(df_val[[TARGET_COL]])
X_test  = scaler_X.transform(df_test[FEATURE_COLS])
y_test  = scaler_y.transform(df_test[[TARGET_COL]])

# PyTorch tensors
def to_tensor(*arrays, dtype=torch.float32):
    return [torch.tensor(a, dtype=dtype).to(DEVICE) for a in arrays]

Xt_tr, yt_tr = to_tensor(X_train, y_train)
Xt_va, yt_va = to_tensor(X_val,   y_val)
Xt_te, yt_te = to_tensor(X_test,  y_test)

BATCH = 256
train_loader = DataLoader(TensorDataset(Xt_tr, yt_tr), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xt_va, yt_va), batch_size=BATCH)
n_features   = X_train.shape[1]
print(f'n_features = {n_features}')

## 5. Model Architectures

In [ ]:
# =========================================================================
# MODEL 1: MLP — Multi-Layer Perceptron
# =========================================================================
class MLP(nn.Module):
    def __init__(self, in_features, hidden=(128, 256, 256, 128), dropout=0.15):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# =========================================================================
# MODEL 2: LSTM
# =========================================================================
class LSTMModel(nn.Module):
    """Treat each feature as one time-step in a sequence of length n_features."""
    def __init__(self, input_size=1, seq_len=8, hidden=128, n_layers=2, dropout=0.2):
        super().__init__()
        self.seq_len = seq_len
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden,
                            num_layers=n_layers, batch_first=True,
                            dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        # x: (B, n_features) → reshape to (B, seq_len, 1)
        x = x.unsqueeze(-1)   # (B, n_features, 1)
        out, _ = self.lstm(x)  # (B, n_features, hidden)
        return self.fc(out[:, -1, :])  # last step


# =========================================================================
# MODEL 3: CNN-LSTM
# =========================================================================
class CNNLSTMModel(nn.Module):
    """1-D convolution extracts local feature patterns, LSTM captures sequence context."""
    def __init__(self, n_features=8, cnn_channels=64, kernel=3, lstm_hidden=128,
                 n_lstm_layers=2, dropout=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, cnn_channels, kernel_size=kernel, padding=kernel//2),
            nn.ReLU(),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=kernel, padding=kernel//2),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(input_size=cnn_channels, hidden_size=lstm_hidden,
                            num_layers=n_lstm_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x = x.unsqueeze(1)         # (B, 1, n_features)
        x = self.conv(x)           # (B, cnn_channels, n_features)
        x = x.permute(0, 2, 1)    # (B, n_features, cnn_channels)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# =========================================================================
# MODEL 4: PINN — Physics-Informed Neural Network
#   Data loss  : MSE on labeled RF data
#   Physics loss: residual of dRF/d(pv) = (1-fw(Sw)) approximation
#                 Simplified: dRF/dPVI ≥ 0 and d²RF/dPVI² ≤ 0  (concavity)
# =========================================================================
class PINNModel(nn.Module):
    def __init__(self, in_features, hidden=(128, 256, 256, 128), dropout=0.1):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]   # Tanh for smooth gradients
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def physics_loss(self, x_raw, pv_col_idx=0):
        """Penalize negative dRF/dPVI and positive d²RF/dPVI² (violates BL physics)."""
        x = x_raw.clone().requires_grad_(True)
        rf = self.net(x)  # (B, 1)

        grad1 = torch.autograd.grad(rf.sum(), x, create_graph=True)[0]
        dpv = grad1[:, pv_col_idx:pv_col_idx+1]   # dRF/dPVI

        grad2 = torch.autograd.grad(dpv.sum(), x, create_graph=True)[0]
        d2pv = grad2[:, pv_col_idx:pv_col_idx+1]  # d²RF/dPVI²

        # BL physics: dRF/dPVI ≥ 0  &  d²RF/dPVI² ≤ 0
        loss_mono    = torch.relu(-dpv).pow(2).mean()   # penalise negative slope
        loss_concave = torch.relu(d2pv).pow(2).mean()   # penalise convexity
        return loss_mono + loss_concave


print('Model classes defined.')

## 6. Training Loop & Utilities

In [ ]:
def train_model(model, train_loader, val_loader, epochs=200,
                lr=1e-3, weight_decay=1e-4, patience=20,
                pinn_lambda=0.0, pinn_pv_col=0):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10,
                                                      factor=0.5, verbose=False)
    criterion = nn.MSELoss()

    history = {'train_loss': [], 'val_loss': [], 'epoch': []}
    best_val = np.inf
    best_state = None
    no_improve = 0

    t0 = time.time()
    for ep in range(1, epochs + 1):
        # ---- training ----
        model.train()
        tr_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            if pinn_lambda > 0:
                loss = loss + pinn_lambda * model.physics_loss(xb, pinn_pv_col)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item() * xb.size(0)
        tr_loss /= len(train_loader.dataset)

        # ---- validation ----
        model.eval()
        va_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                pred = model(xb)
                va_loss += criterion(pred, yb).item() * xb.size(0)
        va_loss /= len(val_loader.dataset)

        scheduler.step(va_loss)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['epoch'].append(ep)

        if va_loss < best_val:
            best_val = va_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {ep}  (best val MSE={best_val:.6f})')
                break

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    print(f'  Training done in {elapsed:.1f}s | best val MSE = {best_val:.6f}')
    return history


def evaluate(model, X_t, y_raw, scaler_y):
    """Return predictions and metrics in original (un-scaled) space."""
    model.eval()
    with torch.no_grad():
        y_pred_scaled = model(X_t).cpu().numpy()
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    y_true = scaler_y.inverse_transform(y_raw.cpu().numpy())
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return y_pred.flatten(), y_true.flatten(), {'RMSE': rmse, 'MAE': mae, 'R2': r2}


results  = {}   # store {name: {'metrics': ..., 'history': ..., 'y_pred': ..., 'y_true': ...}}
print('Training utilities ready.')

## 7. Train All Models

In [ ]:
EPOCHS   = 300
LR       = 5e-4
PATIENCE = 30

# ------------------------------------------------------------------
# 7.1  MLP
# ------------------------------------------------------------------
print('=' * 50)
print('Training MLP …')
mlp = MLP(n_features).to(DEVICE)
print(f'  Params: {sum(p.numel() for p in mlp.parameters()):,}')
hist_mlp = train_model(mlp, train_loader, val_loader,
                        epochs=EPOCHS, lr=LR, patience=PATIENCE)
y_pred_mlp, y_true, m_mlp = evaluate(mlp, Xt_te, yt_te, scaler_y)
results['MLP'] = {'metrics': m_mlp, 'history': hist_mlp,
                   'y_pred': y_pred_mlp, 'y_true': y_true}
print(f'  Test  R²={m_mlp["R2"]:.4f}  RMSE={m_mlp["RMSE"]:.4f}  MAE={m_mlp["MAE"]:.4f}')

In [ ]:
# ------------------------------------------------------------------
# 7.2  LSTM
# ------------------------------------------------------------------
print('=' * 50)
print('Training LSTM …')
lstm = LSTMModel(input_size=1, seq_len=n_features, hidden=128, n_layers=2).to(DEVICE)
print(f'  Params: {sum(p.numel() for p in lstm.parameters()):,}')
hist_lstm = train_model(lstm, train_loader, val_loader,
                         epochs=EPOCHS, lr=LR, patience=PATIENCE)
y_pred_lstm, y_true, m_lstm = evaluate(lstm, Xt_te, yt_te, scaler_y)
results['LSTM'] = {'metrics': m_lstm, 'history': hist_lstm,
                    'y_pred': y_pred_lstm, 'y_true': y_true}
print(f'  Test  R²={m_lstm["R2"]:.4f}  RMSE={m_lstm["RMSE"]:.4f}  MAE={m_lstm["MAE"]:.4f}')

In [ ]:
# ------------------------------------------------------------------
# 7.3  CNN-LSTM
# ------------------------------------------------------------------
print('=' * 50)
print('Training CNN-LSTM …')
cnn_lstm = CNNLSTMModel(n_features=n_features).to(DEVICE)
print(f'  Params: {sum(p.numel() for p in cnn_lstm.parameters()):,}')
hist_cnn = train_model(cnn_lstm, train_loader, val_loader,
                        epochs=EPOCHS, lr=LR, patience=PATIENCE)
y_pred_cnn, y_true, m_cnn = evaluate(cnn_lstm, Xt_te, yt_te, scaler_y)
results['CNN-LSTM'] = {'metrics': m_cnn, 'history': hist_cnn,
                        'y_pred': y_pred_cnn, 'y_true': y_true}
print(f'  Test  R²={m_cnn["R2"]:.4f}  RMSE={m_cnn["RMSE"]:.4f}  MAE={m_cnn["MAE"]:.4f}')

In [ ]:
# ------------------------------------------------------------------
# 7.4  PINN
# ------------------------------------------------------------------
print('=' * 50)
print('Training PINN …')
pinn = PINNModel(n_features).to(DEVICE)
print(f'  Params: {sum(p.numel() for p in pinn.parameters()):,}')
# Physics weight λ=0.05 balances data vs BL physics losses
hist_pinn = train_model(pinn, train_loader, val_loader,
                         epochs=EPOCHS, lr=LR, patience=PATIENCE,
                         pinn_lambda=0.05, pinn_pv_col=0)
y_pred_pinn, y_true, m_pinn = evaluate(pinn, Xt_te, yt_te, scaler_y)
results['PINN'] = {'metrics': m_pinn, 'history': hist_pinn,
                    'y_pred': y_pred_pinn, 'y_true': y_true}
print(f'  Test  R²={m_pinn["R2"]:.4f}  RMSE={m_pinn["RMSE"]:.4f}  MAE={m_pinn["MAE"]:.4f}')

## 8. Results Analysis & Visualisation

In [ ]:
# ------------------------------------------------------------------
# 8.1  Metrics summary table
# ------------------------------------------------------------------
rows = []
for name, res in results.items():
    m = res['metrics']
    n_params = sum(p.numel() for p in {
        'MLP': mlp, 'LSTM': lstm, 'CNN-LSTM': cnn_lstm, 'PINN': pinn
    }[name].parameters())
    rows.append({'Model': name, 'R²': m['R2'], 'RMSE': m['RMSE'],
                 'MAE': m['MAE'], 'Params': n_params})

df_metrics = pd.DataFrame(rows).sort_values('R²', ascending=False).reset_index(drop=True)
df_metrics['Rank'] = df_metrics.index + 1
print('\n===  Test-Set Metrics  ===')
print(df_metrics.to_string(index=False))

In [ ]:
# ------------------------------------------------------------------
# 8.2  Training & Validation Loss Curves
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, res) in zip(axes, results.items()):
    h = res['history']
    ax.semilogy(h['epoch'], h['train_loss'], label='Train', color=COLORS[name])
    ax.semilogy(h['epoch'], h['val_loss'],   label='Val',   color=COLORS[name], linestyle='--')
    ax.set_title(f'{name}', fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss (log)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
plt.suptitle('Training & Validation Loss Curves', fontsize=13)
plt.tight_layout()
plt.savefig('plot_04_loss_curves.png', bbox_inches='tight')
plt.show()
print('Saved plot_04_loss_curves.png')

In [ ]:
# ------------------------------------------------------------------
# 8.3  Predicted vs Actual (parity plots)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, res) in zip(axes, results.items()):
    yt = res['y_true']
    yp = res['y_pred']
    m  = res['metrics']
    ax.scatter(yt, yp, alpha=0.25, s=4, color=COLORS[name])
    lim = [min(yt.min(), yp.min()) - 0.01, max(yt.max(), yp.max()) + 0.01]
    ax.plot(lim, lim, 'k--', linewidth=0.8, label='Ideal')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Actual RF')
    ax.set_ylabel('Predicted RF')
    ax.set_title(f'{name}\nR²={m["R2"]:.4f}  RMSE={m["RMSE"]:.4f}')
    ax.legend(fontsize=8)
plt.suptitle('Parity Plots — Predicted vs Actual Recovery Factor', fontsize=13)
plt.tight_layout()
plt.savefig('plot_05_parity.png', bbox_inches='tight')
plt.show()
print('Saved plot_05_parity.png')

In [ ]:
# ------------------------------------------------------------------
# 8.4  Residual distributions
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, res) in zip(axes, results.items()):
    residuals = res['y_pred'] - res['y_true']
    ax.hist(residuals, bins=50, color=COLORS[name], edgecolor='white',
            linewidth=0.3, alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axvline(np.mean(residuals), color='red', linestyle='-', linewidth=1,
               label=f'Mean={np.mean(residuals):.4f}')
    ax.set_xlabel('Residual (Pred − True)')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} Residuals')
    ax.legend(fontsize=8)
plt.suptitle('Residual Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('plot_06_residuals.png', bbox_inches='tight')
plt.show()
print('Saved plot_06_residuals.png')

In [ ]:
# ------------------------------------------------------------------
# 8.5  Residuals vs Actual (heteroscedasticity check)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, res) in zip(axes, results.items()):
    residuals = res['y_pred'] - res['y_true']
    ax.scatter(res['y_true'], residuals, alpha=0.2, s=4, color=COLORS[name])
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Actual RF')
    ax.set_ylabel('Residual')
    ax.set_title(f'{name}')
plt.suptitle('Residuals vs Actual — Heteroscedasticity Check', fontsize=13)
plt.tight_layout()
plt.savefig('plot_07_residuals_vs_actual.png', bbox_inches='tight')
plt.show()
print('Saved plot_07_residuals_vs_actual.png')

In [ ]:
# ------------------------------------------------------------------
# 8.6  Recovery curve predictions for selected test cases
# ------------------------------------------------------------------
# Pick 6 random test cases and plot predicted vs actual RF vs PVI
selected = rng.choice(test_cases, 6, replace=False)

# Gather all test set case indices
test_case_arr  = df_test['case'].values
pv_test_arr    = df_test['pv_inj'].values
rf_test_arr    = df_test['RF'].values

# Model predictions (already computed)
pred_dict = {name: res['y_pred'] for name, res in results.items()}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, case_id in zip(axes.flat, selected):
    mask = test_case_arr == case_id
    pv   = pv_test_arr[mask]
    rf   = rf_test_arr[mask]
    order = np.argsort(pv)
    ax.plot(pv[order], rf[order], 'ko-', markersize=3, linewidth=1.5,
            label='True', zorder=5)
    for name, preds in pred_dict.items():
        ax.plot(pv[order], preds[mask][order], '-', linewidth=1.5,
                color=COLORS[name], label=name, alpha=0.8)
    ax.set_xlabel('PV Injected')
    ax.set_ylabel('RF')
    ax.set_title(f'Test Case {case_id}')
    ax.legend(fontsize=7)

plt.suptitle('Recovery Curve Predictions — Selected Test Cases', fontsize=13)
plt.tight_layout()
plt.savefig('plot_08_rf_curves.png', bbox_inches='tight')
plt.show()
print('Saved plot_08_rf_curves.png')

In [ ]:
# ------------------------------------------------------------------
# 8.7  Metric comparison bar chart (R², RMSE, MAE)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
model_names = list(results.keys())
colors_list = [COLORS[n] for n in model_names]

for ax, metric in zip(axes, ['R2', 'RMSE', 'MAE']):
    vals = [results[n]['metrics'][metric] for n in model_names]
    bars = ax.bar(model_names, vals, color=colors_list, edgecolor='white', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} — Test Set')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    if metric == 'R2':
        ax.set_ylim(0, 1.05)

plt.suptitle('Model Performance Comparison — Test Set', fontsize=13)
plt.tight_layout()
plt.savefig('plot_09_bar_comparison.png', bbox_inches='tight')
plt.show()
print('Saved plot_09_bar_comparison.png')

In [ ]:
# ------------------------------------------------------------------
# 8.8  Radar / Spider chart — multi-metric overview
# ------------------------------------------------------------------
from matplotlib.patches import FancyArrowPatch

# Normalise metrics so higher = better for all dimensions
metric_labels = ['R²', '1-RMSE (norm)', '1-MAE (norm)', 'Efficiency']

max_rmse = max(results[n]['metrics']['RMSE'] for n in model_names)
max_mae  = max(results[n]['metrics']['MAE']  for n in model_names)
max_par  = max(sum(p.numel() for p in m.parameters())
               for m in [mlp, lstm, cnn_lstm, pinn])

def radar_vals(name):
    m = results[name]['metrics']
    n_p = sum(p.numel() for p in {'MLP': mlp, 'LSTM': lstm,
                                    'CNN-LSTM': cnn_lstm, 'PINN': pinn}[name].parameters())
    return [
        m['R2'],
        1 - m['RMSE'] / max_rmse,
        1 - m['MAE']  / max_mae,
        1 - n_p / max_par,   # fewer params = more efficient
    ]

angles = np.linspace(0, 2*np.pi, len(metric_labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for name in model_names:
    vals = radar_vals(name)
    vals += vals[:1]
    ax.plot(angles, vals, '-o', linewidth=2, label=name, color=COLORS[name])
    ax.fill(angles, vals, alpha=0.10, color=COLORS[name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Multi-Metric Radar — Test Set', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig('plot_10_radar.png', bbox_inches='tight')
plt.show()
print('Saved plot_10_radar.png')

In [ ]:
# ------------------------------------------------------------------
# 8.9  Error vs PV Injected (how error evolves over depletion time)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pv_bins    = np.linspace(0, VPM, 20)
pv_centres = 0.5 * (pv_bins[:-1] + pv_bins[1:])
pv_test    = df_test['pv_inj'].values

for ax, metric_key, ylabel in zip(axes, ['abs_err', 'sq_err'],
                                   ['Mean Absolute Error', 'RMSE']):
    for name, res in results.items():
        errs = np.abs(res['y_pred'] - res['y_true'])
        if metric_key == 'sq_err':
            errs = errs**2
        binned = []
        for lo, hi in zip(pv_bins[:-1], pv_bins[1:]):
            mask = (pv_test >= lo) & (pv_test < hi)
            binned.append(np.sqrt(errs[mask].mean()) if metric_key == 'sq_err'
                          else errs[mask].mean())
        ax.plot(pv_centres, binned, '-o', markersize=4, linewidth=1.5,
                label=name, color=COLORS[name])
    ax.set_xlabel('PV Injected')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs PV Injected')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Prediction Error Along Recovery Curve', fontsize=13)
plt.tight_layout()
plt.savefig('plot_11_error_vs_pv.png', bbox_inches='tight')
plt.show()
print('Saved plot_11_error_vs_pv.png')

In [ ]:
# ------------------------------------------------------------------
# 8.10  Sensitivity: RF prediction vs single-feature sweep
# ------------------------------------------------------------------
# Sweep viscosity_ratio with all other features at median
median_vals = scaler_X.transform([df[FEATURE_COLS].median().values])[0]

vr_vals = np.linspace(df['viscosity_ratio'].min(),
                       df['viscosity_ratio'].max(), 200)
vr_col_idx = FEATURE_COLS.index('viscosity_ratio')
pv_col_idx = FEATURE_COLS.index('pv_inj')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# sweep viscosity ratio at mid PV
X_sweep = np.tile(median_vals, (200, 1))
vr_scaled = (vr_vals - df['viscosity_ratio'].min()) / \
            (df['viscosity_ratio'].max() - df['viscosity_ratio'].min())
X_sweep[:, vr_col_idx] = vr_scaled
Xs = torch.tensor(X_sweep, dtype=torch.float32).to(DEVICE)

for name, mdl in [('MLP', mlp), ('LSTM', lstm), ('CNN-LSTM', cnn_lstm), ('PINN', pinn)]:
    mdl.eval()
    with torch.no_grad():
        preds_s = scaler_y.inverse_transform(mdl(Xs).cpu().numpy())
    axes[0].plot(vr_vals, preds_s, label=name, color=COLORS[name], linewidth=2)
axes[0].set_xlabel('Viscosity Ratio (μo/μw)')
axes[0].set_ylabel('Predicted RF')
axes[0].set_title('Sensitivity: RF vs Viscosity Ratio')
axes[0].set_xscale('log')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# sweep PV injected at median params
pv_range = np.linspace(0, VPM, 200)
pv_scaled = pv_range / VPM  # MinMax scaled to [0,1]
X_sweep2 = np.tile(median_vals, (200, 1))
X_sweep2[:, pv_col_idx] = pv_scaled
Xs2 = torch.tensor(X_sweep2, dtype=torch.float32).to(DEVICE)

for name, mdl in [('MLP', mlp), ('LSTM', lstm), ('CNN-LSTM', cnn_lstm), ('PINN', pinn)]:
    mdl.eval()
    with torch.no_grad():
        preds_t = scaler_y.inverse_transform(mdl(Xs2).cpu().numpy())
    axes[1].plot(pv_range, preds_t, label=name, color=COLORS[name], linewidth=2)
axes[1].set_xlabel('PV Injected')
axes[1].set_ylabel('Predicted RF')
axes[1].set_title('Sensitivity: RF vs PV Injected (median reservoir)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('Sensitivity Analysis', fontsize=13)
plt.tight_layout()
plt.savefig('plot_12_sensitivity.png', bbox_inches='tight')
plt.show()
print('Saved plot_12_sensitivity.png')

In [ ]:
# ------------------------------------------------------------------
# 8.11  Cumulative distribution of absolute errors
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
for name, res in results.items():
    abs_err = np.sort(np.abs(res['y_pred'] - res['y_true']))
    cdf = np.arange(1, len(abs_err)+1) / len(abs_err)
    ax.plot(abs_err, cdf, linewidth=2, label=name, color=COLORS[name])
ax.axvline(0.02, color='gray', linestyle='--', linewidth=0.8, label='2% error line')
ax.set_xlabel('Absolute Error (RF)')
ax.set_ylabel('Cumulative Fraction')
ax.set_title('CDF of Absolute Prediction Errors')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plot_13_cdf_errors.png', bbox_inches='tight')
plt.show()
print('Saved plot_13_cdf_errors.png')

In [ ]:
# ------------------------------------------------------------------
# 8.12  PINN physics loss monitoring — dRF/dPVI positivity check
# ------------------------------------------------------------------
# Verify the PINN respects the monotonicity constraint (BL physics)
pv_range_phys = np.linspace(0, VPM, 100)
pv_scaled_phys = pv_range_phys / VPM
X_phys = np.tile(median_vals, (100, 1)).astype(np.float32)
X_phys[:, pv_col_idx] = pv_scaled_phys.astype(np.float32)
Xt_phys = torch.tensor(X_phys, dtype=torch.float32, requires_grad=True).to(DEVICE)

pinn.eval()
rf_phys = pinn(Xt_phys)
grad_phys = torch.autograd.grad(rf_phys.sum(), Xt_phys)[0]
drf_dpv = grad_phys[:, pv_col_idx].detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
rf_phys_np = scaler_y.inverse_transform(rf_phys.detach().cpu().numpy())
axes[0].plot(pv_range_phys, rf_phys_np, color=COLORS['PINN'], linewidth=2)
axes[0].set_xlabel('PV Injected')
axes[0].set_ylabel('Predicted RF')
axes[0].set_title('PINN — RF Curve (median reservoir)')
axes[0].grid(alpha=0.3)

axes[1].plot(pv_range_phys, drf_dpv, color=COLORS['PINN'], linewidth=2)
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].fill_between(pv_range_phys, 0, drf_dpv,
                      where=drf_dpv >= 0, alpha=0.3, color='green', label='Physical (≥0)')
axes[1].fill_between(pv_range_phys, 0, drf_dpv,
                      where=drf_dpv < 0, alpha=0.3, color='red', label='Unphysical (<0)')
axes[1].set_xlabel('PV Injected')
axes[1].set_ylabel('dRF/dPVI (scaled)')
axes[1].set_title('PINN — Monotonicity of RF (BL Physics Check)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

pct_physical = np.mean(drf_dpv >= 0) * 100
print(f'PINN monotonicity compliance: {pct_physical:.1f}% of evaluation points have dRF/dPVI ≥ 0')
plt.suptitle('PINN Physics Compliance — Buckley–Leverett Monotonicity', fontsize=13)
plt.tight_layout()
plt.savefig('plot_14_pinn_physics.png', bbox_inches='tight')
plt.show()
print('Saved plot_14_pinn_physics.png')

## 9. Summary & Conclusions

In [ ]:
# ------------------------------------------------------------------
# 9.1  Final summary dashboard
# ------------------------------------------------------------------
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# Panel A: R² comparison
ax1 = fig.add_subplot(gs[0, 0])
r2_vals = [results[n]['metrics']['R2'] for n in model_names]
bars = ax1.barh(model_names, r2_vals, color=colors_list, edgecolor='white')
for bar, v in zip(bars, r2_vals):
    ax1.text(v - 0.001, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', ha='right', va='center', fontsize=9, color='white', fontweight='bold')
ax1.set_xlim(0, 1)
ax1.set_xlabel('R²')
ax1.set_title('R² Score')

# Panel B: RMSE comparison
ax2 = fig.add_subplot(gs[0, 1])
rmse_vals = [results[n]['metrics']['RMSE'] for n in model_names]
bars2 = ax2.barh(model_names, rmse_vals, color=colors_list, edgecolor='white')
for bar, v in zip(bars2, rmse_vals):
    ax2.text(v + 0.0001, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', ha='left', va='center', fontsize=9)
ax2.set_xlabel('RMSE')
ax2.set_title('RMSE')

# Panel C: MAE comparison
ax3 = fig.add_subplot(gs[0, 2])
mae_vals = [results[n]['metrics']['MAE'] for n in model_names]
bars3 = ax3.barh(model_names, mae_vals, color=colors_list, edgecolor='white')
for bar, v in zip(bars3, mae_vals):
    ax3.text(v + 0.0001, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', ha='left', va='center', fontsize=9)
ax3.set_xlabel('MAE')
ax3.set_title('MAE')

# Panel D: Val loss convergence overlay
ax4 = fig.add_subplot(gs[1, 0:2])
for name, res in results.items():
    h = res['history']
    ax4.semilogy(h['epoch'], h['val_loss'], linewidth=1.5,
                  label=name, color=COLORS[name])
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Val MSE Loss')
ax4.set_title('Validation Loss Convergence (all models)')
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

# Panel E: Metrics table as text
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
table_data = [[n, f'{results[n]["metrics"]["R2"]:.4f}',
               f'{results[n]["metrics"]["RMSE"]:.4f}',
               f'{results[n]["metrics"]["MAE"]:.4f}']
              for n in model_names]
tbl = ax5.table(cellText=table_data,
                colLabels=['Model', 'R²', 'RMSE', 'MAE'],
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 2)
ax5.set_title('Test Set Summary', fontsize=12)

fig.suptitle('Proxy5 — DL Model Comparison for Oil Recovery Prediction', fontsize=14, y=1.01)
plt.savefig('plot_15_summary_dashboard.png', bbox_inches='tight')
plt.show()
print('Saved plot_15_summary_dashboard.png')

In [ ]:
# ------------------------------------------------------------------
# 9.2  Print final ranking
# ------------------------------------------------------------------
print('\n' + '='*60)
print('   PROXY5 — FINAL MODEL RANKING (by R²)')
print('='*60)
print(df_metrics[['Rank', 'Model', 'R²', 'RMSE', 'MAE', 'Params']].to_string(index=False))
print('='*60)

best_model = df_metrics.iloc[0]['Model']
print(f'\n>> Best model: {best_model} '
      f'(R²={df_metrics.iloc[0]["R²"]:.4f}, '
      f'RMSE={df_metrics.iloc[0]["RMSE"]:.4f})')

print('''
NOTES
-----
• MLP      : Fastest to train; strong baseline for tabular reservoir data.
• LSTM     : Captures sequential injection history; beneficial with real time-series.
• CNN-LSTM : Local convolution extracts feature interactions; most expressive hybrid.
• PINN     : Physics-constrained — enforces BL monotonicity; best when data are sparse
             or extrapolation is needed outside training conditions.

Physics reference: Liu et al. Physics of Fluids 37 036622 (2025)
''')